code lib setup

In [2]:
!pip install google-cloud-bigquery pandas pyarrow kagglehub

kaggle set up: download dataset

In [3]:
import kagglehub

kagglehub.login()

path = kagglehub.dataset_download("sobhanmoosavi/us-traffic-congestions-2016-2022")

100%|██████████| 2.31G/2.31G [00:27<00:00, 91.4MB/s]

Extracting files...


Kaggle credentials set.
Kaggle credentials successfully validated.


Move file to somewhere handy

In [7]:
import os
import shutil

source_path = '/root/.cache/kagglehub/datasets/sobhanmoosavi/us-traffic-congestions-2016-2022/versions/3/us_congestion_2016_2022_sample_2m/us_congestion_2016_2022_sample_2m.csv'
destination_path = '/content/us_congestion_2016_2022_sample_2m.csv'

# Ensure the destination directory exists if it's not the root /content
# In this case, /content always exists, so no need for os.makedirs(os.path.dirname(destination_path), exist_ok=True)

try:
    shutil.move(source_path, destination_path)
    print(f"File moved successfully from {source_path} to {destination_path}")
except FileNotFoundError:
    print(f"Error: The file {source_path} was not found.")
except Exception as e:
    print(f"An error occurred: {e}")

File moved successfully from /root/.cache/kagglehub/datasets/sobhanmoosavi/us-traffic-congestions-2016-2022/versions/3/us_congestion_2016_2022_sample_2m/us_congestion_2016_2022_sample_2m.csv to /content/us_congestion_2016_2022_sample_2m.csv


In [9]:
!head -n 21 us_congestion_2016_2022_sample_2m.csv > traffic_sample_20.csv #first 20 rows of data for easier understanding

Clean data to be ready to be uploaded to bigquery

In [10]:
import pandas as pd
import numpy as np

# 1. Load Sample Data
df = pd.read_csv('us_congestion_2016_2022_sample_2m.csv')

# 2. Normalize Column Names (Align with BigQuery naming rules: no spaces or parentheses)
def clean_col_name(name):
    return (name.replace('(', '').replace(')', '')
                .replace('-', '_').replace(' ', '_')
                .replace('/', '_').replace('%', 'pct')
                .replace('.', ''))

df.columns = [clean_col_name(col) for col in df.columns]

# 3. Advanced Transformations

# --- Step A: Solve Mixed Timezone Issues by Extracting Local Time ---

# Option 1: Convert to UTC first, then localize back to "Naive" time
# This ensures consistency across different US regions
df['StartTime_Local'] = pd.to_datetime(df['StartTime'], utc=True).map(lambda x: x.tz_localize(None))
df['EndTime_Local'] = pd.to_datetime(df['EndTime'], utc=True).map(lambda x: x.tz_localize(None))

# Option 2: Strictly preserve the "Hour" as written in the original string (ignoring UTC conversion)
df['StartTime_Local'] = pd.to_datetime(df['StartTime']).apply(lambda x: x.replace(tzinfo=None))
df['EndTime_Local'] = pd.to_datetime(df['EndTime']).apply(lambda x: x.replace(tzinfo=None))

# --- Step B: Safely Extract Features Using .dt Accessor ---
df['Start_Hour'] = df['StartTime_Local'].dt.hour
df['Day_of_Week'] = df['StartTime_Local'].dt.day_name()
df['Is_Weekend'] = df['StartTime_Local'].dt.dayofweek >= 5

# Define Rush Hour periods
df['Is_Peak'] = df['Start_Hour'].apply(lambda x: 1 if (7 <= x <= 9 or 16 <= x <= 19) else 0)

# Calculate Duration (Absolute time difference in minutes)
df['Duration_mins'] = (pd.to_datetime(df['EndTime'], utc=True) -
                        pd.to_datetime(df['StartTime'], utc=True)).dt.total_seconds() / 60

# --- Step C: Prepare for BigQuery Upload (Standardize to UTC) ---
df['StartTime_UTC'] = pd.to_datetime(df['StartTime'], utc=True)
df['EndTime_UTC'] = pd.to_datetime(df['EndTime'], utc=True)

# Remove temporary "Local" columns to prevent BigQuery data type errors
df_final = df.drop(columns=['StartTime_Local', 'EndTime_Local'])

print("✅ Successfully resolved timezone issues and extracted local hour features!")
print(df_final[['StartTime_UTC', 'Start_Hour', 'Is_Peak']].head())

# Group Weather Conditions (Reduce dimensionality and increase statistical significance)
def group_weather(condition):
    if pd.isna(condition): return 'Other'
    cond = str(condition).lower()
    if 'rain' in cond or 'drizzle' in cond: return 'Rain'
    if 'snow' in cond or 'ice' in cond or 'freezing' in cond: return 'Snow'
    if 'cloudy' in cond or 'overcast' in cond: return 'Cloudy'
    if 'clear' in cond or 'fair' in cond: return 'Clear'
    if 'fog' in cond or 'mist' in cond or 'haze' in cond: return 'Fog'
    return 'Other'

df['Weather_Grouped'] = df['Weather_Conditions'].apply(group_weather)

# Numerical Encoding for Congestion Speed (Speed_Rank: Higher value means slower traffic)
speed_map = {'Fast': 1, 'Moderate': 2, 'Slow': 3}
df['Speed_Rank'] = df['Congestion_Speed'].map(speed_map)

# Fill missing numerical values with 0 to maintain BigQuery Schema stability
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].fillna(0)

# 4. Save Cleaned CSV
output_file = 'us_congestion_2016_2022_sample_2m_cleaned.csv'
df.to_csv(output_file, index=False)

print(f"✅ Cleaned CSV created successfully: {output_file}")
print(df[['ID', 'Severity', 'Duration_mins', 'Weather_Grouped', 'Is_Peak']].head())

/tmp/ipykernel_5303/2703215224.py:26: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df['StartTime_Local'] = pd.to_datetime(df['StartTime']).apply(lambda x: x.replace(tzinfo=None))
/tmp/ipykernel_5303/2703215224.py:27: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df['EndTime_Local'] = pd.to_datetime(df['EndTime']).apply(lambda x: x.replace(tzinfo=None))


✅ Successfully resolved timezone issues and extracted local hour features!
              StartTime_UTC  Start_Hour  Is_Peak
0 2016-12-21 00:19:00+00:00        19.0        1
1 2018-11-16 22:18:00+00:00        17.0        1
2 2021-02-19 01:32:00+00:00        20.0        0
3 2020-11-13 13:06:00+00:00         8.0        1
4 2017-08-24 13:54:00+00:00         9.0        1
✅ Cleaned CSV created successfully: us_congestion_2016_2022_sample_2m_cleaned.csv
           ID  Severity  Duration_mins Weather_Grouped  Is_Peak
0  C-14344128         2      14.783333           Clear        1
1  C-32285069         0      50.466667          Cloudy        1
2  C-14213642         0      49.533333           Clear        0
3  C-29674072         0      42.366667            Rain        1
4  C-24044478         1      79.316667          Cloudy        1


In [11]:
!ls -lh us_congestion_2016_2022_sample_2m_cleaned.csv #confirming the file size of cleaned csv

-rw-r--r-- 1 root root 1001M Apr  6 05:21 us_congestion_2016_2022_sample_2m_cleaned.csv


Upload data to gcp big query

In [12]:
import kagglehub
import pandas as pd
import os
from google.cloud import bigquery
from google.colab import userdata

# --- 2. Configure GCP Information ---
# If running locally, specify the path to your service account key file
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/cosc301 cred.json"

project_id = userdata.get('GCP_PROJECT_ID') # Get project ID from Colab secrets
dataset_id = 'traffic_data'
table_id = 'us_congestion_records'
table_ref = f"{project_id}.{dataset_id}.{table_id}"

# --- 3. Initialize BigQuery Client ---
client = bigquery.Client(project=project_id)

# --- Create Dataset if it doesn't exist ---
try:
    client.get_dataset(dataset_id)
    print(f"Dataset {dataset_id} already exists.")
except Exception:
    print(f"Creating dataset {dataset_id}...")
    dataset = bigquery.Dataset(f"{project_id}.{dataset_id}")
    dataset.location = "US"  # Set the geographic location for the dataset
    dataset = client.create_dataset(dataset, timeout=30)  # Make an API request.
    print(f"Created dataset {client.project}.{dataset.dataset_id}")

# --- 4. Read and Upload Data ---
# Note: Since the data volume may be large, consider reading in chunks or uploading the file directly
csv_path = '/content/us_congestion_2016_2022_sample_2m_cleaned.csv'
print(f"Reading and uploading: {csv_path}")

# Using pandas to assist with the upload (the most intuitive method)
# Fix: Explicitly parse date columns when reading the CSV
df = pd.read_csv(csv_path, parse_dates=['StartTime','EndTime','StartTime_UTC', 'EndTime_UTC','WeatherTimeStamp'])

# Clean column names to be BigQuery compatible
def clean_col_name(col_name):
    return col_name.replace('(', '').replace(')', '').replace('-', '_').replace(' ', '_')

df.columns = [clean_col_name(col) for col in df.columns]

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE",
    # set fields as TIMESTAMP
    schema=[
        bigquery.SchemaField("StartTime", "TIMESTAMP"),
        bigquery.SchemaField("EndTime", "TIMESTAMP"),
        bigquery.SchemaField("StartTime_UTC", "TIMESTAMP"),
        bigquery.SchemaField("EndTime_UTC", "TIMESTAMP"),
        bigquery.SchemaField("WeatherTimeStamp", "TIMESTAMP"),

    ],
    # autoDetect for others
    autodetect=True,
)
job = client.load_table_from_dataframe(df, table_ref, job_config=job_config)
job.result()  # Wait for the job to complete

print(f"Success! Data has been uploaded to {table_ref}")

Dataset traffic_data already exists.
Reading and uploading: /content/us_congestion_2016_2022_sample_2m_cleaned.csv
Success! Data has been uploaded to cosc301-492411.traffic_data.us_congestion_records
